<a href="https://colab.research.google.com/github/vencov/FAV_upsidedown/blob/main/TubeAcoustics/EX_tube_acoustics_GUI_PH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Simulation of the 1D acoustic field in narrow tubes
### Python/Jupyter port of `TubeAcoustics3.m` (Petr Honzík, CTU in Prague)

This reproduces the original MATLAB GUI as an interactive Jupyter notebook
using `ipywidgets`. It solves the plane-wave acoustic field in a narrow
cylindrical tube including **viscous and thermal boundary-layer losses**
(the low-reduced-frequency / Zwikker–Kosten-type model), for a chosen
terminal condition, and lets you sweep frequency and geometry interactively.

Run all cells; the last cell renders the interactive control panel
(sliders / dropdowns) with live-updating plots, just like the original GUI.

**Note:** static previews of interactive Jupyter widgets don't render outside
a live kernel — open this notebook in Jupyter and run it to get the sliders.


In [1]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import jv
import ipywidgets as widgets
from ipywidgets import interactive_output, VBox, HBox, Layout
from IPython.display import display

%matplotlib inline



## Physical model

Air/tube parameters and the governing equations, following the original
`Tube_Zin_TF` and `Tube_px_vx_tx` MATLAB functions:

- $k_v = \sqrt{\omega\rho/\mu}\,(1-i)/\sqrt{2}$, $k_h = \sqrt{\omega\rho C_p/\lambda_h}\,(1-i)/\sqrt{2}$
  — viscous / thermal boundary-layer wavenumbers.
- $F_v, F_h$ — cross-section-averaged velocity / temperature correction
  factors (Kirchhoff/Zwikker–Kosten theory), from $J_0, J_1$ Bessel functions.
- $\chi = k_0\sqrt{\dfrac{\gamma-(\gamma-1)F_h}{F_v}}$ — complex propagation
  wavenumber for pressure.
- The terminal specific impedance $Z_s = p(L)/v(L)$ depends on the chosen
  end condition (isothermal rigid wall, infinite tube, closed cavity, or a
  user-specified numerical impedance).
- $p(x) = A\big(e^{-i\chi x} + B e^{i\chi x}\big)$, with $A, B$ fixed by the
  input condition (velocity or pressure at $x=0$) and $Z_s$ at $x=L$.



## Interactive panel

Controls mirror the original GUI: tube length/radius, terminal condition
(with its extra parameter shown only when relevant), input condition
(velocity or pressure), the frequency range/resolution, and a slider for the
"current" frequency at which the spatial/radial profiles are shown (marked
by a dotted vertical line on the frequency-domain plots).


In [5]:


import numpy as np
import matplotlib.pyplot as plt
from scipy.special import jv
import ipywidgets as widgets
from ipywidgets import interactive_output, VBox, HBox, Layout
from IPython.display import display

%matplotlib inline


def default_params():
    return dict(
        c0=340.0,        # sound speed [m/s]
        rho=1.18,        # air density [kg/m^3]
        mu=1.83e-5,      # shear dynamic viscosity [Pa*s]
        lambda_h=24.4e-3,# thermal conductivity [W/(m*K)]
        Cp=1010.0,       # specific heat at const. pressure per unit mass [J/(kg*K)]
        gamma=1.4,       # specific heat ratio [-]
        beta=340.38,     # (dP/dT)_V
    )

def _derived(params):
    p = dict(params)
    p['lh'] = p['lambda_h'] / (p['rho'] * p['c0'] * p['Cp'])  # thermal characteristic length
    return p

def _Fv_Fh(omega, R, p):
    kv = np.sqrt(omega * p['rho'] / p['mu']) * (1 - 1j) / np.sqrt(2)
    kh = np.sqrt(omega * p['rho'] * p['Cp'] / p['lambda_h']) * (1 - 1j) / np.sqrt(2)
    Fv = 1 - (2 * jv(1, kv * R) / (kv * R * jv(0, kv * R)))
    Fh = 1 - (2 * jv(1, kh * R) / (kh * R * jv(0, kh * R)))
    return kv, kh, Fv, Fh

def _terminal_impedance(kind, omega, k0, CHI, Fv, R, p, Vc, Zvalue):
    if kind == 'infinite tube':
        return (omega * p['rho']) / (Fv * CHI)
    elif kind == 'cavity':
        return np.pi * R**2 * p['rho'] * p['c0']**2 / (1j * omega * Vc)
    elif kind == 'numerical value':
        return np.full_like(omega, Zvalue, dtype=complex)
    else:  # 'isothermal rigid wall'
        return p['rho'] * p['c0'] * np.sqrt(2) / ((1 + 1j) * np.sqrt(k0) * (p['gamma'] - 1) * np.sqrt(p['lh']))


def tube_zin_tf(L, R, freq, params, terminal='isothermal rigid wall', v_in=1e-3, Vc=1e-7, Zvalue=4.8e3):
    '''Vectorized over freq. Returns Z_in(f), TF(f)=p(L)/p(0), p_in(f), tau_in(f).'''
    p = _derived(params)
    omega = 2*np.pi*freq
    k0 = omega/p['c0']
    kv, kh, Fv, Fh = _Fv_Fh(omega, R, p)
    CHI = k0*np.sqrt((p['gamma']-(p['gamma']-1)*Fh)/Fv)
    Zs = _terminal_impedance(terminal, omega, k0, CHI, Fv, R, p, Vc, Zvalue)

    B = np.exp(-2j*CHI*L)*(-1+Zs*CHI*Fv/(omega*p['rho']))/(1+Zs*CHI*Fv/(omega*p['rho']))
    A = v_in*omega*p['rho']/(CHI*Fv*(1-B))

    p_in = A*(1+B)
    p_L = A*(np.exp(-1j*CHI*L) + B*np.exp(1j*CHI*L))
    tau_in = p_in*(p['gamma']-1)*Fh/(p['beta']*p['gamma'])
    Z_in = omega*p['rho']*(1+B)/(CHI*Fv*(1-B))
    TF = p_L/p_in
    return Z_in, TF, p_in, tau_in


def tube_px_vx_tx(L, R, curr_freq, x_coord, params, terminal='isothermal rigid wall',
                   input_type='input velocity', v_in=1e-3, p_in_user=1.0, Vc=1e-7, Zvalue=4.8e3):
    '''Single-frequency spatial profiles + radial boundary-layer profiles.'''
    p = _derived(params)
    omega = 2*np.pi*curr_freq
    k0 = omega/p['c0']
    kv, kh, Fv, Fh = _Fv_Fh(omega, R, p)
    CHI = k0*np.sqrt((p['gamma']-(p['gamma']-1)*Fh)/Fv)
    Zs = _terminal_impedance(terminal, np.array([omega]), np.array([k0]), np.array([CHI]),
                              np.array([Fv]), R, p, Vc, Zvalue)[0]

    B = np.exp(-2j*CHI*L)*(-1+Zs*CHI*Fv/(omega*p['rho']))/(1+Zs*CHI*Fv/(omega*p['rho']))
    if input_type == 'input pressure':
        A = p_in_user/(1+B)
    else:
        A = v_in*omega*p['rho']/(CHI*Fv*(1-B))

    p_x = A*(np.exp(-1j*CHI*x_coord) + B*np.exp(1j*CHI*x_coord))
    v_x = -1j*CHI*A*(-np.exp(-1j*CHI*x_coord) + B*np.exp(1j*CHI*x_coord))*Fv/(1j*omega*p['rho'])
    tau_x = p_x*(p['gamma']-1)*Fh/(p['beta']*p['gamma'])

    r = np.linspace(-R, R, 400)
    fv_r = 1 - jv(0, kv*r)/jv(0, kv*R)
    dv = np.sqrt(2*p['mu']/(p['rho']*omega))
    fh_r = 1 - jv(0, kh*r)/jv(0, kh*R)
    dh = np.sqrt(2*p['lambda_h']/(p['rho']*p['Cp']*omega))
    return p_x, v_x, tau_x, fv_r, fh_r, dv, dh, r




params = default_params()

w_L = widgets.FloatText(value=20e-3, description='L [m]:', step=1e-3, layout=Layout(width='180px'))
w_R = widgets.FloatText(value=1e-3, description='R [m]:', step=1e-4, layout=Layout(width='180px'))

w_terminal = widgets.Dropdown(
    options=['isothermal rigid wall', 'infinite tube', 'cavity', 'numerical value'],
    value='isothermal rigid wall', description='Terminal:', layout=Layout(width='260px'))
w_Vc = widgets.FloatText(value=1e-7, description='Vc [m^3]:', layout=Layout(width='180px'))
w_Zvalue = widgets.FloatText(value=4.8e3, description='Zs [kg/(s.m^2)]:', layout=Layout(width='200px'))

w_input = widgets.Dropdown(options=['input velocity', 'input pressure'],
                            value='input velocity', description='Input:', layout=Layout(width='220px'))
w_vin = widgets.FloatText(value=1e-3, description='v(0) [m/s]:', layout=Layout(width='180px'))
w_pin = widgets.FloatText(value=1.0, description='p(0) [Pa]:', layout=Layout(width='180px'))

w_fmin = widgets.FloatText(value=10.0, description='f_min [Hz]:', layout=Layout(width='180px'))
w_fmax = widgets.FloatText(value=3e4, description='f_max [Hz]:', layout=Layout(width='180px'))
w_npts = widgets.IntText(value=500, description='# freq pts:', layout=Layout(width='180px'))

w_freq_log = widgets.FloatLogSlider(value=1000, base=10, min=1, max=np.log10(3e4), step=0.001,
                                     description='Current f [Hz]:', readout_format='.1f',
                                     layout=Layout(width='500px'))


def _update_visibility(*args):
    w_Vc.layout.display = None if w_terminal.value == 'cavity' else 'none'
    w_Zvalue.layout.display = None if w_terminal.value == 'numerical value' else 'none'
    w_vin.layout.display = None if w_input.value == 'input velocity' else 'none'
    w_pin.layout.display = None if w_input.value == 'input pressure' else 'none'

w_terminal.observe(_update_visibility, 'value')
w_input.observe(_update_visibility, 'value')
_update_visibility()


def plot_all(L, R, terminal, Vc, Zvalue, input_type, v_in, p_in_user, fmin, fmax, npts, curr_freq):
    freq = np.logspace(np.log10(fmin), np.log10(fmax), int(npts))
    x_coord = np.linspace(0, L, 100)

    Zin, TF, p_in_f, tau_in = tube_zin_tf(L, R, freq, params, terminal, v_in, Vc, Zvalue)
    p_x, v_x, tau_x, fv_r, fh_r, dv, dh, r = tube_px_vx_tx(
        L, R, curr_freq, x_coord, params, terminal, input_type, v_in, p_in_user, Vc, Zvalue)

    fig = plt.figure(figsize=(14, 12))
    gs = fig.add_gridspec(4, 3, width_ratios=[1, 1, 0.5], hspace=0.9, wspace=0.35)

    # --- TF magnitude / phase ---
    ax = fig.add_subplot(gs[0, 0])
    TF_dB = 20*np.log10(np.abs(TF))
    ax.semilogx(freq, TF_dB)
    ax.axvline(curr_freq, color='g', linestyle=':')
    ax.set_xlabel('frequency [Hz]'); ax.set_ylabel('|TF| [dB]')
    ax.set_title('Transfer function TF = p(L)/p(0)')
    ax.grid(True)

    ax = fig.add_subplot(gs[1, 0])
    ax.semilogx(freq, np.unwrap(np.angle(TF)))
    ax.axvline(curr_freq, color='g', linestyle=':')
    ax.set_xlabel('frequency [Hz]'); ax.set_ylabel('phase(TF) [rad]')
    ax.grid(True)

    # --- Zin magnitude / phase ---
    ax = fig.add_subplot(gs[0, 1])
    ax.loglog(freq, np.abs(Zin))
    ax.axvline(curr_freq, color='g', linestyle=':')
    ax.set_xlabel('frequency [Hz]'); ax.set_ylabel('|Z_in| [kg/(s.m^2)]')
    ax.set_title('Input impedance Z_in = p(0)/v(0)')
    ax.grid(True)

    ax = fig.add_subplot(gs[1, 1])
    ax.semilogx(freq, np.unwrap(np.angle(Zin)))
    ax.axvline(curr_freq, color='g', linestyle=':')
    ax.set_xlabel('frequency [Hz]'); ax.set_ylabel('phase(Z_in) [rad]')
    ax.grid(True)

    # --- schematic ---
    ax = fig.add_subplot(gs[0, 2])
    ax.plot([0, L], [R, R], 'k', lw=1.5)
    ax.plot([0, L], [-R, -R], 'k', lw=1.5)
    if terminal == 'isothermal rigid wall':
        ax.plot([L, L], [-R, R], 'k', lw=1.5)
    elif terminal == 'numerical value':
        ax.plot([L, L], [-R, R], 'k:', lw=1.5)
    elif terminal == 'infinite tube':
        ax.annotate('', xy=(L*1.15, R), xytext=(L, R), arrowprops=dict(arrowstyle='-', ls=':'))
        ax.annotate('', xy=(L*1.15, -R), xytext=(L, -R), arrowprops=dict(arrowstyle='-', ls=':'))
    elif terminal == 'cavity':
        cw = 0.15*L
        ax.plot([L, L+cw], [R, R], 'k', lw=1.5)
        ax.plot([L, L+cw], [-R, -R], 'k', lw=1.5)
        ax.plot([L+cw, L+cw], [-R, R], 'k', lw=1.5)
    ax.annotate('', xy=(L*0.28, 0), xytext=(0, 0), arrowprops=dict(arrowstyle='->'))
    ax.text(-L*0.03, -R*1.7, 'p(0)', ha='center')
    ax.text(L*0.15, R*1.3, 'v(0)', ha='center')
    ax.text(L*1.0, -R*1.7, 'p(L)', ha='center')
    ax.set_xlim(-L*0.1, L*1.3); ax.set_ylim(-2.2*R, 2.2*R)
    ax.set_title('Tube schematic'); ax.axis('off')

    # --- p(x), v(x), tau(x) ---
    ax = fig.add_subplot(gs[1, 2])
    ax.plot(x_coord, np.real(p_x), label='Re(p)')
    ax.plot(x_coord, np.imag(p_x), label='Im(p)')
    ax.set_xlabel('x [m]'); ax.set_ylabel('p(x) [Pa]')
    ax.set_title(f'p(x) @ {curr_freq:.1f} Hz'); ax.legend(fontsize=8); ax.grid(True)

    ax = fig.add_subplot(gs[2, 0])
    ax.plot(x_coord, np.real(v_x), label='Re(v)')
    ax.plot(x_coord, np.imag(v_x), label='Im(v)')
    ax.set_xlabel('x [m]'); ax.set_ylabel('v(x) [m/s]')
    ax.set_title(f'Particle velocity @ {curr_freq:.1f} Hz'); ax.legend(fontsize=8); ax.grid(True)

    ax = fig.add_subplot(gs[2, 1])
    ax.plot(x_coord, np.real(tau_x), label=r'Re($\tau$)')
    ax.plot(x_coord, np.imag(tau_x), label=r'Im($\tau$)')
    ax.set_xlabel('x [m]'); ax.set_ylabel(r'$\tau(x)$ [K]')
    ax.set_title(f'Temperature variation @ {curr_freq:.1f} Hz'); ax.legend(fontsize=8); ax.grid(True)

    ax = fig.add_subplot(gs[2, 2])
    ax.plot(np.real(fv_r), r*1e3, label='Re')
    ax.plot(np.imag(fv_r), r*1e3, label='Im')
    ax.set_ylabel('r [mm]'); ax.set_title(f'Velocity profile\n$d_v$={dv*1e6:.1f} um')
    ax.legend(fontsize=8); ax.grid(True)

    ax = fig.add_subplot(gs[3, 2])
    ax.plot(np.real(fh_r), r*1e3, label='Re')
    ax.plot(np.imag(fh_r), r*1e3, label='Im')
    ax.set_ylabel('r [mm]'); ax.set_title(f'Temperature profile\n$d_h$={dh*1e6:.1f} um')
    ax.legend(fontsize=8); ax.grid(True)

    plt.tight_layout()
    plt.show()


out = interactive_output(plot_all, dict(
    L=w_L, R=w_R, terminal=w_terminal, Vc=w_Vc, Zvalue=w_Zvalue,
    input_type=w_input, v_in=w_vin, p_in_user=w_pin,
    fmin=w_fmin, fmax=w_fmax, npts=w_npts, curr_freq=w_freq_log,
))

controls = VBox([
    HBox([w_L, w_R]),
    HBox([w_terminal, w_Vc, w_Zvalue]),
    HBox([w_input, w_vin, w_pin]),
    HBox([w_fmin, w_fmax, w_npts]),
    w_freq_log,
])

display(VBox([controls, out]))
